# Simulating a user driving every registered planner through MCP, logged as real OCEL 2.0

This notebook plays the part of a human exploring `autofde_lab`'s catalog through its real
MCP server (`autofde_lab.fabric.mcp.create_server`) -- not the happy path of one
hand-picked domain/solver pair (see `tests/fabric/test_dspy_mcp_planner_loop_chicago.py`
for that), but what actually happens when someone tries every registered domain against
every solver the catalog says is compatible with it, over a real `fastmcp.Client` connected
to a real server.

Every real MCP call in this notebook -- `decision_catalog`, `decision_match`, and every
`decision_solve` -- is recorded as a real OCEL 2.0 event, using the repo's own
`autofde_lab.ocel.log.OcelLog` (validated, not just written). The result, saved to
`notebooks/artifacts/mcp_user_simulation.ocel.json`, is a real object-centric event log of
one simulated session, objects = `MCPSession` / `Domain` / `Solver`.

**Measured this session, before writing this notebook** (see
`~/.claude/plans/read-docs-autofde-explore-md-89-lines-ancient-hippo.md`): of 26 registered
domains, 21 refuse to construct with default (zero) arguments -- `SKD-FABRIC-006`, a real,
typed refusal, not a crash. Only 5 construct with defaults and have compatible solvers, for
117 real domain×solver pairs total. That asymmetry -- most of the catalog genuinely refuses a
naive default-arguments call -- **is** the non-happy path this notebook is asked to show, not
a defect in it.

No timeout mechanism exists anywhere in this repo for a `solve()` call (confirmed this
session against `fabric/coverage.py::_run_solver`, the only existing precedent for "run every
solver"), and several of the 117 pairs are real RL-training solvers
(`RayRLlib`/`StableBaseline`/`AugmentedRandomSearch`/`MaxentIRL`) with no bound on training
time. Each `decision_solve` call below runs in its own subprocess
(`scripts/mcp_solve_one_pair.py`) under a real, OS-enforced `subprocess.run(timeout=...)`, so
a hung solver becomes a real `TIMEOUT` event instead of hanging this notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

from fastmcp import Client

from autofde_lab.fabric.mcp import create_server
from autofde_lab.fabric.service import DecisionFabric
from autofde_lab.ocel.log import OcelLog

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOLVE_ONE_PAIR_SCRIPT = REPO_ROOT / "scripts" / "mcp_solve_one_pair.py"
OUTPUT_PATH = REPO_ROOT / "notebooks" / "artifacts" / "mcp_user_simulation.ocel.json"

MAX_STEPS = 15
PER_PAIR_TIMEOUT_S = 15

assert SOLVE_ONE_PAIR_SCRIPT.exists(), f"missing {SOLVE_ONE_PAIR_SCRIPT}"

fabric = DecisionFabric()
server = create_server(fabric)

## Step 1: real `decision_catalog` over a real MCP client

In-memory transport (`Client(server)`), but the real `fastmcp` protocol layer: `initialize`,
`tools/list`, `tools/call` -- the same code path a remote client speaking MCP over stdio would
drive.

In [ ]:
async def get_catalog() -> dict:
    async with Client(server) as client:
        result = await client.call_tool("decision_catalog", {})
        return result.data


catalog = await get_catalog()
domains = list(catalog["domains"])
solvers_catalog = list(catalog["solvers"])
print(f"{len(domains)} registered domains, {len(solvers_catalog)} registered solvers")

## Step 2: build the OCEL log's object set

One `MCPSession` object for this run, one `Domain` object per registered domain, one `Solver`
object per registered solver. Declared up front, per `OcelLog.append_event`'s own contract
(objects must exist before an event links to them).

In [ ]:
from autofde_lab.ocel.model import (
    OcelAttribute,
    OcelAttributeValue,
    OcelObject,
)

SESSION_ID = "session-mcp-user-simulation-1"

log = OcelLog.new().with_objects(
    OcelObject(
        SESSION_ID,
        "MCPSession",
        (OcelAttribute("server", OcelAttributeValue.string("scikit-decide-fabric")),),
    ),
    *(
        OcelObject(
            f"domain-{name}",
            "Domain",
            (OcelAttribute("name", OcelAttributeValue.string(name)),),
        )
        for name in domains
    ),
    *(
        OcelObject(
            f"solver-{name}",
            "Solver",
            (OcelAttribute("name", OcelAttributeValue.string(name)),),
        )
        for name in solvers_catalog
    ),
)

log = log.append_event(
    "evt-catalog",
    "decision_catalog",
    [SESSION_ID],
    timestamp_ns=time.time_ns(),
    attributes={
        "domain_count": OcelAttributeValue.integer(len(domains)),
        "solver_count": OcelAttributeValue.integer(len(solvers_catalog)),
    },
)
print(len(log.objects), "objects declared")

## Step 3: interleaved match + solve for the domains known to construct with no arguments

**A real defect found while first running this notebook, not designed around in advance:**
attempting `decision_match("FlightPlanningDomain")` (or any domain requiring real construction
arguments this notebook doesn't supply) imports `cartopy`/`pyproj` as a side effect of loading
the domain module, even though construction itself then refuses (`SKD-FABRIC-006`). Once that
import has happened in this kernel process, *every subsequent* `subprocess.run()` call in this
process crashes on macOS -- `libproj` registers a `pthread_atfork` handler that touches
`os_log`/SQLite cleanup, which is unsafe to run in a forked child before `exec()`. All 117
`decision_solve` subprocess calls failed with the same crash trace the first time this notebook
ran, in a single run where the full 26-domain match sweep (Step 4 below) ran *before* any
solving.

The fix applied here: do the real, subprocess-based `decision_solve` calls for the domains
already measured this session to construct with zero arguments (`Maze`, `SimpleGridWorld`,
`MasterMind`, `RockPaperScissors`, `GymWidthDomain`) **first**, interleaved with their own real
`decision_match` call -- before this kernel process has touched any domain that imports
`cartopy`. The full 26-domain match sweep, including `FlightPlanningDomain` and every other
domain that refuses construction, runs in Step 4, after all real subprocess solving is done.

In [ ]:
from autofde_lab.fabric.bounded_exec import run_subprocess_bounded
from autofde_lab.ocel.mcp_session import append_tool_call_event


async def solve_one_pair(domain: str, solver: str) -> dict:
    outcome = await run_subprocess_bounded(
        [sys.executable, str(SOLVE_ONE_PAIR_SCRIPT), domain, solver, str(MAX_STEPS)],
        timeout_s=PER_PAIR_TIMEOUT_S,
        env={**os.environ, "OBJC_DISABLE_INITIALIZE_FORK_SAFETY": "YES"},
    )
    if outcome.standing == "TIMEOUT":
        return {"standing": "TIMEOUT", "elapsed_s": outcome.elapsed_s}
    if outcome.standing == "ERROR":
        return {
            "standing": "ERROR",
            "elapsed_s": outcome.elapsed_s,
            "detail": outcome.stderr[-1000:],
        }
    try:
        payload = json.loads(outcome.stdout.strip().splitlines()[-1])
    except (json.JSONDecodeError, IndexError):
        return {
            "standing": "ERROR",
            "elapsed_s": outcome.elapsed_s,
            "detail": "unparseable stdout",
        }
    if "error" in payload:
        return {
            "standing": "REFUSED",
            "elapsed_s": outcome.elapsed_s,
            "detail": payload["error"],
        }
    return {
        "standing": payload.get("standing", "UNKNOWN"),
        "elapsed_s": outcome.elapsed_s,
        "steps": len(payload.get("steps", [])),
        "receipt_sha256": payload.get("receipt_sha256"),
    }


def log_solve_event(log, domain_name: str, solver_name: str, outcome: dict):
    n_prior = len(
        [
            e
            for e in log.events
            if e.id.startswith(f"evt-solve-{domain_name}-{solver_name}")
        ]
    )
    return append_tool_call_event(
        log,
        event_id=f"evt-solve-{domain_name}-{solver_name}-{n_prior}",
        activity="decision_solve",
        object_ids=[SESSION_ID, f"domain-{domain_name}", f"solver-{solver_name}"],
        outcome=outcome,
    )


def log_match_event(log, domain_name: str, outcome: dict):
    if outcome["ok"]:
        compatible = outcome["data"].get("compatible_solvers", [])
        log = append_tool_call_event(
            log,
            event_id=f"evt-match-{domain_name}",
            activity="decision_match",
            object_ids=[SESSION_ID, f"domain-{domain_name}"],
            outcome={"standing": "MATCHED", "compatible_solver_count": len(compatible)},
        )
        return log, compatible
    log = append_tool_call_event(
        log,
        event_id=f"evt-match-{domain_name}",
        activity="decision_match",
        object_ids=[SESSION_ID, f"domain-{domain_name}"],
        outcome={"standing": "REFUSED", "error": outcome["error"]},
    )
    return log, []


async def match_one(name: str) -> dict:
    async with Client(server) as client:
        try:
            result = await client.call_tool(
                "decision_match", {"domain": name, "use_cache": False}
            )
            return {"ok": True, "data": result.data}
        except Exception as exc:  # noqa: BLE001 -- a real refusal is data, not a crash
            return {"ok": False, "error": f"{type(exc).__name__}: {exc}"}


KNOWN_ZERO_ARG_CONSTRUCTIBLE = [
    "Maze",
    "SimpleGridWorld",
    "MasterMind",
    "RockPaperScissors",
    "GymWidthDomain",
]
# Order matters here for the fork-safety reason explained above -- these run, and are fully
# solved, before this process touches any domain that imports cartopy/pyproj.
safe_domains = [d for d in KNOWN_ZERO_ARG_CONSTRUCTIBLE if d in domains]
assert safe_domains, "none of the known-safe domains are in this catalog -- investigate"

solve_outcomes: list[tuple[str, str, dict]] = []
pairs: list[tuple[str, str]] = []
sweep_start = time.time()

for domain_name in safe_domains:
    outcome = await match_one(domain_name)
    log, compatible = log_match_event(log, domain_name, outcome)
    for solver_name in compatible:
        pairs.append((domain_name, solver_name))
        result = await solve_one_pair(domain_name, solver_name)
        solve_outcomes.append((domain_name, solver_name, result))
        log = log_solve_event(log, domain_name, solver_name, result)
        if len(solve_outcomes) % 10 == 0:
            print(
                f"  {len(solve_outcomes)} pairs solved so far, {time.time() - sweep_start:.0f}s elapsed"
            )

print(
    f"Phase A (safe domains): {len(pairs)} real pairs solved in {time.time() - sweep_start:.1f}s"
)

## Step 4: real `decision_match` for every remaining registered domain

Now safe to attempt the domains that need real constructor arguments this notebook doesn't
supply (`FlightPlanningDomain` among them) -- all real solving already happened in Step 3, so
there is nothing left for a poisoned fork to break. Each refusal is logged as a real event,
not swallowed.

In [ ]:
remaining_domains = [d for d in domains if d not in safe_domains]

t0 = time.time()
for name in remaining_domains:
    outcome = await match_one(name)
    log, compatible = log_match_event(log, name, outcome)
    # None of these are expected to have compatible solvers (they all fail construction with
    # default arguments) -- but if the catalog ever changes and one does construct, solve its
    # pairs too, for real, same as Phase A. Documented, not silently dropped.
    for solver_name in compatible:
        result = await solve_one_pair(name, solver_name)
        solve_outcomes.append((name, solver_name, result))
        pairs.append((name, solver_name))
        log = log_solve_event(log, name, solver_name, result)

print(f"remaining {len(remaining_domains)} domains matched in {time.time() - t0:.1f}s")
refused_domains = sum(
    1
    for d in domains
    if any(
        e.id == f"evt-match-{d}"
        and any(
            a.key == "standing" and a.value.to_json() == "REFUSED" for a in e.attributes
        )
        for e in log.events
    )
)
print(
    f"{refused_domains}/{len(domains)} domains refused construction with default arguments"
)
print(f"{len(pairs)} total real domain×solver pairs attempted across both phases")

## Step 5: validate, persist, and report honestly

`OcelLog.validate()` enforces OCPQ Definition 2's structural laws (every event links a real
object, no dangling references, exactly one type per entity) before anything is trusted or
saved -- a log that fails validation is a bug in this notebook, not a result to report.

In [ ]:
log = log.validate()
print("OCEL log structurally valid:", True)
print("digest:", log.digest())

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(log.to_ocel2_json(), indent=2))
print("saved:", OUTPUT_PATH)

from collections import Counter

standings = Counter(outcome["standing"] for _, _, outcome in solve_outcomes)
print("\nsolve outcomes across", len(pairs), "real pairs:")
for standing, count in standings.most_common():
    print(f"  {standing:10s} {count}")

print(f"\n{refused_domains}/{len(domains)} domains refused decision_match construction")
print(
    "This is a bounded, non-happy-path simulation: most of the catalog genuinely",
    "refuses default arguments, and a real fraction of solve attempts are",
    "TIMEOUT/REFUSED/ERROR, not silently hidden as if everything were ALIVE.",
)